# Student Engagement — EDA & Preprocessing
### *Branch B · Facial Engagement Classification*

---

| Field | Details |
|-------|---------|
| **Project** | Non-Invasive Cognitive Load and Student Engagement Detection |
| **Author** | Yasini Mandara Karunanayake |
| **RGU ID** | 2313473  **IIT ID** 20221151 |
| **Dataset** | [Student Concentration Image Dataset](https://www.kaggle.com/datasets/programmer3/student-concentration-image-dataset) |


---

## Notebook Overview

This notebook implements **Stage 1 of Branch B** of the multimodal pipeline.
It downloads the Kaggle Student Concentration Image Dataset, conducts a five-part EDA, applies the full preprocessing pipeline (resize → grayscale → CLAHE → HOG), performs a block-based leakage-free train/test split, balances the training set with SMOTE, and exports scaled HOG feature arrays for model training.

| Step | Description |
|------|-------------|
| 1 | Environment setup & library imports |
| 2 | Dataset download & metadata catalogue |
| 3 | Exploratory Data Analysis (5 visualisations) |
| 4 | Image quality checks (corrupt, duplicate, blur) |
| 5 | Preprocessing pipeline (resize → grayscale → CLAHE → HOG) |
| 6 | Block-based GroupShuffleSplit (leakage-free 80/20 split) |
| 7 | HOG feature visualisation |
| 8 | SMOTE class balancing |
| 9 | StandardScaler fitting & feature export |


## 1. Environment Setup

Install packages not available by default in the Google Colab environment.

- **scikit-image** - Image I/O, HOG feature extraction, CLAHE preprocessing
- **imbalanced-learn** - SMOTE oversampling for class balancing
- **opencv-python-headless** - Grayscale conversion, Laplacian-based blur detection


In [ ]:
# Install extra packages not included in the default Colab environment
!pip install -q imbalanced-learn scikit-image opencv-python-headless
print('Dependencies installed.')

Dependencies installed.


In [ ]:
# Standard library imports
import os
import warnings
import math
import hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from skimage import io, color, transform, exposure
from skimage.feature import hog
from skimage.util import img_as_float
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

print('All imports successful.')

All imports successful.


## 2. Download Dataset via Kaggle API

The **Kaggle Student Concentration Image Dataset** (2,120 images across six sub-classes) is downloaded directly into the Colab environment via the Kaggle API.
This dataset forms the training data for facial engagement classification of the multimodal pipeline.


In [ ]:
!pip install -q kagglehub
print('kagglehub installed')


kagglehub installed


In [ ]:
# Kaggle credentials
import os
os.environ['KAGGLE_USERNAME'] = 'username'
os.environ['KAGGLE_KEY']      = 'kaggle_key'
print('Kaggle credentials set')

Kaggle credentials set


In [ ]:
import kagglehub

# Download the dataset
KAGGLE_DATASET_PATH = kagglehub.dataset_download(
    'programmer3/student-concentration-image-dataset'
)

print('Path to dataset files:', KAGGLE_DATASET_PATH)
print()

# Show top-level folder structure
import os
for root, dirs, files_ in os.walk(KAGGLE_DATASET_PATH):
    level = root.replace(KAGGLE_DATASET_PATH, '').count(os.sep)
    if level < 3:
        print('  ' * level + os.path.basename(root) + '/')
        if level == 2 and files_:
            print('  ' * (level + 1) + f'[{len(files_)} files, e.g. {files_[0]}]')


100%|██████████| 38.4M/38.4M [00:04<00:00, 8.48MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/programmer3/student-concentration-image-dataset/versions/1

1/
  Student Dataset/
    Student-engagement-dataset/


In [ ]:
# Search from kagglehub download root for the engagement dataset folder
DATA_ROOT = Path(KAGGLE_DATASET_PATH)

DATA_DIR = None
for p in DATA_ROOT.rglob('Student-engagement-dataset'):
    if p.is_dir():
        DATA_DIR = p
        break

if DATA_DIR is None:
    DATA_DIR = DATA_ROOT
    print('Student-engagement-dataset folder not found — using root:', DATA_ROOT)

# Output directory — all .npy arrays and .pkl files will be saved here
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Image size constants: HOG is computed on 64×64 greyscale images
IMG_SIZE        = (64, 64)
DISPLAY_SIZE    = (128, 128)

# 80% to training, 20% to test
TEST_SPLIT      = 0.20
RANDOM_STATE    = 42

# Only read files with these image extensions; ignore anything else
IMG_EXTENSIONS  = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

# Six-to-three class remapping
CLASS_MAP = {
    'confused':     'high',
    'focused':      'high',
    'frustrated':   'medium',
    'bored':        'medium',
    'drowsy':       'low',
    'looking away': 'low',
    'looking_away': 'low',
}

# Ordered engagement levels and their integer encodings (0=low, 1=medium, 2=high)
ENGAGEMENT_LABELS = ['low', 'medium', 'high']
LABEL_ENC = {lbl: i for i, lbl in enumerate(ENGAGEMENT_LABELS)}
LABEL_DEC = {i: lbl for lbl, i in LABEL_ENC.items()}

# Colours used in all plots: red=low, orange=medium, green=high
PALETTE = {'low': '#e74c3c', 'medium': '#f39c12', 'high': '#2ecc71'}
ORIG_PALETTE = {
    'Confused':     '#3498db',
    'Focused':      '#2ecc71',
    'Frustrated':   '#e67e22',
    'Bored':        '#e74c3c',
    'Drowsy':       '#9b59b6',
    'Looking Away': '#1abc9c',
}

print('Configuration ready')
print(f'   Dataset path : {DATA_DIR}')
print(f'   Output path  : {OUTPUT_DIR}')


Configuration ready
   Dataset path : /root/.cache/kagglehub/datasets/programmer3/student-concentration-image-dataset/versions/1/Student Dataset/Student-engagement-dataset
   Output path  : /content/output


## 2. Dataset Discovery & Metadata

Walk the dataset directory tree to build a complete image catalogue. Each record captures the file path, original six-class label, remapped three-class engagement level, and integer label encoding.
Consecutive images from the same recording are grouped into temporal blocks in preparation for the block split `GroupShuffleSplit` applied in Section 6.

In [ ]:
records = []

# Walk the folder tree and build one record per image file
for parent in sorted(DATA_DIR.iterdir()):
    if not parent.is_dir():
        continue
    for cls_folder in sorted(parent.iterdir()):
        if not cls_folder.is_dir():
            continue
        cls_key    = cls_folder.name.lower()
# Map the subfolder name to the remapped engagement level
        engagement = CLASS_MAP.get(cls_key)
        if engagement is None:
            print(f'  Skipped unrecognised folder: {cls_folder.name}')
            continue

        display_name = 'Looking Away' if 'looking' in cls_key else cls_folder.name

        for img_path in cls_folder.iterdir():
            if img_path.suffix.lower() in IMG_EXTENSIONS:
                stat = img_path.stat()
# Store file path, label, and file-size metadata for each image
                records.append({
                    'path':           str(img_path),
                    'filename':       img_path.name,
                    'extension':      img_path.suffix.lower(),
                    'file_size_kb':   round(stat.st_size / 1024, 2),
                    'original_class': display_name,
                    'parent_folder':  parent.name,
                    'engagement':     engagement,
                    'label':          LABEL_ENC[engagement],
                })

# Build the dataframe and shuffle rows to remove folder-ordering bias
df = (pd.DataFrame(records)
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True))

print(f'Total images collected : {len(df):,}')
print()
print(df.groupby(['parent_folder', 'original_class', 'engagement'])
        .size().reset_index(name='count').to_string(index=False))

# Assign temporal block IDs for GroupShuffleSplit
BLOCK_SIZE = 10  # frames per block

# Sort by class and filename so consecutive frames from the same video sit adjacent
df = df.sort_values(['original_class', 'filename']).reset_index(drop=True)
df['seq_within_class'] = df.groupby('original_class').cumcount()

# Group every BLOCK_SIZE consecutive frames into one block ID
# All frames in a block will always go to the same split (train or test)
df['block_id'] = (df['original_class'] + '_block_'
                  + (df['seq_within_class'] // BLOCK_SIZE).astype(str))

n_blocks = df['block_id'].nunique()
print(f'Block IDs assigned  : {n_blocks} blocks ({BLOCK_SIZE} frames each)')
print(f'Total images        : {len(df):,}')
print()
print(df.groupby('original_class')['block_id'].nunique()
        .rename('num_blocks').to_string())


Total images collected : 2,120

parent_folder original_class engagement  count
      Engaged       Confused       high    369
      Engaged        Focused       high    347
      Engaged     Frustrated     medium    360
  Not Engaged          Bored     medium    358
  Not Engaged         Drowsy        low    263
  Not Engaged   Looking Away        low    423
Block IDs assigned  : 214 blocks (10 frames each)
Total images        : 2,120

original_class
Bored           36
Confused        37
Drowsy          27
Focused         35
Frustrated      36
Looking Away    43


In [ ]:
# Read a random subset to profile native sizes
sample_paths = df['path'].sample(min(300, len(df)), random_state=RANDOM_STATE).tolist()

# Collect height, width, and channel count for each sampled image
heights, widths, channels = [], [], []
for p in sample_paths:
    try:
        img = io.imread(p)
# Handle both grayscale (2-D) and colour (3-D) images
        if img.ndim == 2:
            h, w = img.shape;    c = 1
        else:
            h, w, c = img.shape
        heights.append(h);  widths.append(w);  channels.append(c)
    except Exception:
        pass

# Wrap the collected dimension lists into a dataframe for easy summary stats
df_dims = pd.DataFrame({'height': heights, 'width': widths, 'channels': channels})

print('Native image dimension statistics (sampled from 300 images):')
print(df_dims.describe().round(1))
print()
print('Channel distribution:')
print(df_dims['channels'].value_counts().to_string())

Native image dimension statistics (sampled from 300 images):
       height   width  channels
count   300.0   300.0     300.0
mean    708.5  1147.0       3.0
std      28.1   325.7       0.0
min     640.0   352.0       3.0
25%     720.0  1280.0       3.0
50%     720.0  1280.0       3.0
75%     720.0  1280.0       3.0
max     720.0  1280.0       3.0

Channel distribution:
channels
3    300
